In [59]:
import pandas as pd
import json
import re

# --- 1. CONFIGURATION ---
hierarchy_paths = {
    "RSD": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/rsd_cm_hierarchy.csv",
    "PSD": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/psd_cm_hierarchy.csv",
    "SNC": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/snc_cm_hierarchy.csv",
    "TNM": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/tnm_cm_hierarchy.csv"
}

cm_data_paths = {
    "RSD": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/rsd_cm_data.csv",
    "PSD": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/psd_cm_data.csv",
    "SNC": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/snc_cm_data.csv",
    "TNM": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/SAP Integration/tnm_cm_data.csv"
}

def load_hierarchy_data(file_path):
    df = pd.read_csv(file_path)
    return df

def load_cm_data(file_path):
    df = pd.read_csv(file_path)
    return df

### RSD

In [60]:
rsd_hierarchy_df = load_hierarchy_data(hierarchy_paths["RSD"])

rsd_hierarchy_df.drop(columns=["System"], inplace=True)

rsd_hierarchy_df.rename(columns={
    "Sub System": "sub_system",
    "Root Cause": "root_cause",
    }, inplace=True)

for col in rsd_hierarchy_df.columns:
    rsd_hierarchy_df[col] = rsd_hierarchy_df[col].replace("\n", " ", regex=True)
    rsd_hierarchy_df[col] = rsd_hierarchy_df[col].replace("\r", " ", regex=True)
    rsd_hierarchy_df[col] = rsd_hierarchy_df[col].ffill()

cols_to_clean = ['sub_system', 'root_cause']
for col in cols_to_clean:
    rsd_hierarchy_df[col] = rsd_hierarchy_df[col].str.replace(r'\s+', ' ', regex=True).str.strip()

rsd_hierarchy_df['department'] = 'RSDM'

# rsd_hierarchy_df['sub_system_char_count'] = rsd_hierarchy_df['sub_system'].str.len()
# rsd_hierarchy_df['root_cause_char_count'] = rsd_hierarchy_df['root_cause'].str.len()

# print("--- Top 10 Longest sub_system entries ---")
# display(rsd_hierarchy_df.nlargest(10, 'sub_system_char_count')[['sub_system', 'sub_system_char_count']])

# print("\n--- Top 10 Longest root_cause entries ---")
# display(rsd_hierarchy_df.nlargest(10, 'root_cause_char_count')[['root_cause', 'root_cause_char_count']])

In [61]:
rsd_cm_data = load_cm_data(cm_data_paths["RSD"])

rsd_cm_data.rename(columns={
    "sub_system": "sub_system",
    "Root Cause": "root_cause",
    "Train_number": "train_number",
    "Car_Position": "car_position",
    "notification_date": "created_on",
    }, inplace=True)

clean_cols = ['sub_system', 'root_cause', 'cause_items', 'activity_items']
for col in clean_cols:
    rsd_cm_data[col] = rsd_cm_data[col].replace("\n", " ", regex=True)
    rsd_cm_data[col] = rsd_cm_data[col].replace("\r", " ", regex=True)
    rsd_cm_data[col] = rsd_cm_data[col].str.replace(r'\s+', ' ', regex=True).str.strip()

rsd_cm_data['remarks'] = (
    "Cause Items: " + rsd_cm_data['cause_items'].astype(str) + "|| Activity Items: " + rsd_cm_data['activity_items'].astype(str)
)

col_order = ['work_order_no', 'notification_no', 'created_on', 'functional_location', 'train_number', 'car_position', 'sub_system', 'root_cause', 'remarks', 'date_closed']

rsd_cm_data = rsd_cm_data[col_order]
rsd_cm_data.head()

,work_order_no,notification_no,created_on,functional_location,train_number,car_position,sub_system,root_cause,remarks,date_closed
0,4000468245,11956356,1/6/2022,RSV022-CAR05-BOG-01,RSV022,ECA1,Bogie System (BOG),BCU faulty,Cause Items: Request to replace bogie 1 assemb...,1/6/2022
1,4000468465,11956402,1/6/2022,RSV023-CAR12-LVS,RSV023,ECA4,Low Voltage System (LVS),UOF,"Cause Items: 1345Hrs.RSV23 C2309 TSA/Up, TO YI...",2/6/2022
2,4000467985,11952610,1/6/2022,RSV027-CAR26-PPS-CC3,RSV027,ICA2,Primary Power System (PPS),UOF,Cause Items: Request to replace current collec...,1/6/2022
3,4000468466,11956574,1/6/2022,RSV027-CAR28-PRS-TR6,RSV027,ECA4,Propulsion System (PRS),UOF,"Cause Items: 1951hrs, RSV 27 Car 2728 RAN/Dn, ...",2/6/2022
4,4000469005,11957292,2/6/2022,RSV022-CAR06-BOG-03,RSV022,ICA2,Vehicle Management System (VMS),TPMS Sensor Faulty,"Cause Items: 2321hrs, RSV 22 Car 2208 BAS/Dn, ...",3/6/2022


In [62]:
import numpy as np

checks = {
    'sub_system': 'sub_system',
    'root_cause': 'root_cause'
}

for target_col, ref_col in checks.items():
    valid_values = rsd_hierarchy_df[ref_col].unique()
    
    check_col_name = f'check_{target_col}'
    rsd_cm_data[check_col_name] = rsd_cm_data[target_col].isin(valid_values)
    
    num_false = (~rsd_cm_data[check_col_name]).sum()
    num_true = rsd_cm_data[check_col_name].sum()
    
    print(f"--- Results for {target_col} ---")
    print(f"Total rows: {len(rsd_cm_data)}")
    print(f"Valid: {num_true} | Invalid: {num_false}")
    if num_false > 0:
        print(f"Invalid samples: {rsd_cm_data.loc[~rsd_cm_data[check_col_name], target_col].unique()[:5]}")
    print("-" * 30)

rsd_hierarchy_df.to_csv("system_hierarchy/RSD_Hierarchy.csv", sep="|", index=False)
rsd_cm_data.head(10).to_csv("system_hierarchy/RSD_CM_Data.csv", sep="|", index=False)

# Batch processing to save memory and avoid potential issues with very large files
num_files = 3  
batches = np.array_split(rsd_cm_data, num_files)

print(f"Total rows: {len(rsd_cm_data)} || Splitting into {num_files} files...\n")

for i, batch_df in enumerate(batches):
    filename = f"system_hierarchy/RSD_CM_Data_batch{i+1}.csv"
    batch_df.to_csv(filename, sep="|", index=False)
    
    print(f"File: {filename} | Rows: {len(batch_df)}")

--- Results for sub_system ---
Total rows: 14026
Valid: 13995 | Invalid: 31
Invalid samples: ['Others/Unmapped']
------------------------------
--- Results for root_cause ---
Total rows: 14026
Valid: 14026 | Invalid: 0
------------------------------
Total rows: 14026 || Splitting into 3 files...

File: system_hierarchy/RSD_CM_Data_batch1.csv | Rows: 4676
File: system_hierarchy/RSD_CM_Data_batch2.csv | Rows: 4675
File: system_hierarchy/RSD_CM_Data_batch3.csv | Rows: 4675


d:\Santai Coding\Pradigma - Digital Maintenance Portal\Coding\python.notebook.extraction\venv\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


### PSD

In [63]:
psd_hierarchy_df = load_hierarchy_data(hierarchy_paths["PSD"])

psd_hierarchy_df.rename(columns={
    "System": "system",
    "Sub system": "sub_system",
    "Sub-Sub System": "sub_sub_system",
    "Root Cause": "root_cause",
    }, inplace=True)

for col in psd_hierarchy_df.columns:
    psd_hierarchy_df[col] = psd_hierarchy_df[col].replace("\n", " ", regex=True)
    psd_hierarchy_df[col] = psd_hierarchy_df[col].replace("\r", " ", regex=True)
    psd_hierarchy_df[col] = psd_hierarchy_df[col].str.replace(r'\s+', ' ', regex=True).str.strip()
    psd_hierarchy_df[col] = psd_hierarchy_df[col].ffill()

psd_hierarchy_df['department'] = 'WS10'
psd_hierarchy_df.head()

# cols_to_check = ['system', 'sub_system', 'sub_sub_system', 'root_cause']

# for col in cols_to_check:
#     count_col = f"{col}_char_count"
    
#     # Calculate character length for the column
#     psd_hierarchy_df[count_col] = psd_hierarchy_df[col].str.len()
    
#     # Display the top 10 most characters for this specific column
#     print(f"--- Top 10 Longest {col} entries ---")
#     display(psd_hierarchy_df.nlargest(10, count_col)[[col, count_col]])
#     print("\n") # Adds a line break between tables

,system,sub_system,sub_sub_system,root_cause,department
0,Generator Set,Volvo 100kVa,AMF Board,"Relay, Timer, loose connection, Under voltage ...",WS10
1,Generator Set,Volvo 100kVa,Engine,"Engine oil, filter & Coolant leak",WS10
2,Generator Set,Volvo 100kVa,Alternator,Insulation is cracked or brittle,WS10
3,Generator Set,Volvo 100kVa,Battery,"Charger card failure, power supply failure, Ba...",WS10
4,Generator Set,Volvo 175kVa,AMF Board,"Relay, Timer, loose connection, Under voltage ...",WS10


In [64]:
psd_cm_data = load_cm_data(cm_data_paths["PSD"])

psd_cm_data.columns = psd_cm_data.columns.str.lower().str.replace(' ', '_')

for col in psd_cm_data.columns:
    psd_cm_data[col] = psd_cm_data[col].replace("\n", " ", regex=True)
    psd_cm_data[col] = psd_cm_data[col].replace("\r", " ", regex=True)
    psd_cm_data[col] = psd_cm_data[col].ffill()

psd_cm_data.rename(columns={
    "sub-sub_system": "sub_sub_system",
    "location": "station",
    }, inplace=True)


psd_cm_data['remarks'] = (
    "Incident Description: " + psd_cm_data['incident_description'].astype(str) + "Failure: " + psd_cm_data['failure'].astype(str)
)

psd_cm_data['created_on'] = pd.to_datetime(
    psd_cm_data['date'].astype(str) + ' ' + psd_cm_data['time'].astype(str),
    format='mixed',
    errors='coerce'
)

col_order = ['work_order_no', 'notification_no', 'created_on', 'station', 'system', 'sub_system', 'sub_sub_system', 'root_cause', 'remarks']
psd_cm_data = psd_cm_data[col_order]

psd_cm_data['work_order_no'] = pd.to_numeric(psd_cm_data['work_order_no'], errors='coerce').fillna(0).astype(int)
psd_cm_data['notification_no'] = pd.to_numeric(psd_cm_data['notification_no'], errors='coerce').fillna(0).astype(int)

psd_cm_data["sub_sub_system"] = psd_cm_data["sub_sub_system"].fillna("Others/Unmapped")
psd_cm_data["root_cause"] = psd_cm_data["root_cause"].fillna("Others/Unmapped")

psd_cm_data = psd_cm_data[psd_cm_data['work_order_no'].notnull() & (psd_cm_data['work_order_no'] != 0)]
psd_cm_data_incomplete = psd_cm_data[psd_cm_data['work_order_no'].isnull() | (psd_cm_data['work_order_no'] == 0)]

psd_cm_data_incomplete.shape

(0, 9)

In [65]:
import numpy as np

checks = {
    'system': 'system',
    'sub_system': 'sub_system',
    'sub_sub_system': 'sub_sub_system',
    'root_cause': 'root_cause'
}

for target_col, ref_col in checks.items():
    psd_cm_data[target_col] = psd_cm_data[target_col].replace("Others/Unmapped", "Others")
    
    valid_values = psd_hierarchy_df[ref_col].unique()
    
    check_col_name = f'check_{target_col}'
    psd_cm_data[check_col_name] = psd_cm_data[target_col].isin(valid_values)
    
    num_false = (~psd_cm_data[check_col_name]).sum()
    num_true = psd_cm_data[check_col_name].sum()
    
    print(f"--- Results for {target_col} ---")
    print(f"Total rows: {len(psd_cm_data)}")
    print(f"Valid: {num_true} | Invalid: {num_false}")
    if num_false > 0:
        print(f"Invalid samples: {psd_cm_data.loc[~psd_cm_data[check_col_name], target_col].unique()[:5]}")
    print("-" * 30)
    

psd_hierarchy_df.to_csv("system_hierarchy/PSD_Hierarchy.csv", sep="|", index=False)
psd_cm_data.to_csv("system_hierarchy/PSD_CM_Data.csv", sep="|", index=False)

--- Results for system ---
Total rows: 1066
Valid: 1036 | Invalid: 30
Invalid samples: ['Others']
------------------------------
--- Results for sub_system ---
Total rows: 1066
Valid: 1036 | Invalid: 30
Invalid samples: ['Others']
------------------------------
--- Results for sub_sub_system ---
Total rows: 1066
Valid: 451 | Invalid: 615
Invalid samples: ['Others']
------------------------------
--- Results for root_cause ---
Total rows: 1066
Valid: 451 | Invalid: 615
Invalid samples: ['Others']
------------------------------


### SNC

In [66]:
snc_hierarchy_df = load_hierarchy_data(hierarchy_paths["SNC"])

snc_hierarchy_df.rename(columns={
    "System": "system",
    "Sub System": "sub_system",
    "Sub Sub System": "sub_sub_system",
    "Root Cause": "root_cause",
    }, inplace=True)

for col in snc_hierarchy_df.columns:
    snc_hierarchy_df[col] = snc_hierarchy_df[col].replace("\n", " ", regex=True)
    snc_hierarchy_df[col] = snc_hierarchy_df[col].replace("\r", " ", regex=True)
    
    snc_hierarchy_df[col] = snc_hierarchy_df[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    
    snc_hierarchy_df[col] = snc_hierarchy_df[col].ffill()

snc_hierarchy_df['department'] = 'WS20'

# cols_to_check = ['system', 'sub_system', 'sub_sub_system', 'root_cause']

# for col in cols_to_check:
#     count_col = f"{col}_char_count"
    
#     # Calculate character length
#     snc_hierarchy_df[count_col] = snc_hierarchy_df[col].str.len()
    
#     # Display the top 10 longest entries
#     print(f"--- Top 10 Longest {col} entries (SNC) ---")
#     display(snc_hierarchy_df.nlargest(10, count_col)[[col, count_col]])
#     print("\n")

In [67]:
snc_cm_data = load_cm_data(cm_data_paths["SNC"])

snc_cm_data.rename(columns={
    "system": "system",
    "sub system": "sub_system",
    "sub-sub system": "sub_sub_system",
    "root cause": "root_cause",
    "location": "station",
    }, inplace=True)

for col in snc_cm_data.columns:
    snc_cm_data[col] = snc_cm_data[col].replace("\n", " ", regex=True)
    snc_cm_data[col] = snc_cm_data[col].replace("\r", " ", regex=True)

snc_cm_data['created_on'] = pd.to_datetime(
    snc_cm_data['date'].astype(str) + ' ' + snc_cm_data['time'].astype(str),
    format='mixed',
    errors='coerce'
)

col_order = ['work_order_no', 'notification_no', 'created_on', 'station', 'system', 'sub_system', 'sub_sub_system', 'root_cause']
snc_cm_data = snc_cm_data[col_order]

snc_cm_data['work_order_no'] = pd.to_numeric(snc_cm_data['work_order_no'], errors='coerce').fillna(0).astype(int)
snc_cm_data['notification_no'] = pd.to_numeric(snc_cm_data['notification_no'], errors='coerce').fillna(0).astype(int)

snc_cm_data = snc_cm_data[snc_cm_data['work_order_no'].notnull() & (snc_cm_data['work_order_no'] != 0)]
snc_cm_data_incomplete = snc_cm_data[snc_cm_data['work_order_no'].isnull() | (snc_cm_data['work_order_no'] == 0)]

snc_cm_data.head()

,work_order_no,notification_no,created_on,station,system,sub_system,sub_sub_system,root_cause
0,4000445168,11896478,2022-01-15 06:04:29,BFZ,Others/Unmapped,Others/Unmapped,Others/Unmapped,Others/Unmapped
1,4000445166,11896848,2022-01-16 05:55:39,HAH,INTERLOCKING SYSTEM,MEI/MESD,FRONTAL PING/ESD,Fan problem
2,4000452440,11912924,2022-02-27 05:12:57,BFZ,CTC System,KVM Switch,Power Adapter,System intermittent
3,4000452439,11913141,2022-02-28 08:42:29,MKU,CTC System,KVM Switch,VGA Cable,System intermittent
4,4000454634,11918824,2022-03-06 07:38:44,HAH,PA - PIS,IPPA,UTP Cable,System hang


In [68]:
import numpy as np

checks = {
    'system': 'system',
    'sub_system': 'sub_system',
    'sub_sub_system': 'sub_sub_system',
    'root_cause': 'root_cause'
}

for target_col, ref_col in checks.items():
    snc_cm_data[target_col] = snc_cm_data[target_col].replace("Others/Unmapped", "Others")
    
    valid_values = snc_hierarchy_df[ref_col].unique()
    
    check_col_name = f'check_{target_col}'
    snc_cm_data[check_col_name] = snc_cm_data[target_col].isin(valid_values)
    
    num_false = (~snc_cm_data[check_col_name]).sum()
    num_true = snc_cm_data[check_col_name].sum()
    
    print(f"--- Results for {target_col} ---")
    print(f"Total rows: {len(snc_cm_data)}")
    print(f"Valid: {num_true} | Invalid: {num_false}")
    if num_false > 0:
        print(f"Invalid samples: {snc_cm_data.loc[~snc_cm_data[check_col_name], target_col].unique()[:5]}")
    print("-" * 30)

snc_hierarchy_df.to_csv("system_hierarchy/SNC_Hierarchy.csv", sep="|", index=False)
snc_cm_data.to_csv("system_hierarchy/SNC_CM_Data.csv", sep="|", index=False)

--- Results for system ---
Total rows: 137
Valid: 114 | Invalid: 23
Invalid samples: ['Others']
------------------------------
--- Results for sub_system ---
Total rows: 137
Valid: 114 | Invalid: 23
Invalid samples: ['Others']
------------------------------
--- Results for sub_sub_system ---
Total rows: 137
Valid: 114 | Invalid: 23
Invalid samples: ['Others']
------------------------------
--- Results for root_cause ---
Total rows: 137
Valid: 114 | Invalid: 23
Invalid samples: ['Others']
------------------------------


### TNM

In [69]:
tnm_hierarchy_df = load_hierarchy_data(hierarchy_paths["TNM"])

tnm_hierarchy_df.rename(columns={
    "System": "system",
    "Sub System": "sub_system",
    "Sub Sub System": "sub_sub_system",
    }, inplace=True)

for col in tnm_hierarchy_df.columns:
    tnm_hierarchy_df[col] = tnm_hierarchy_df[col].replace("\n", " ", regex=True)
    tnm_hierarchy_df[col] = tnm_hierarchy_df[col].replace("\r", " ", regex=True)

    tnm_hierarchy_df[col] = tnm_hierarchy_df[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    
    tnm_hierarchy_df[col] = tnm_hierarchy_df[col].ffill()

tnm_hierarchy_df['department'] = 'TNMD'

cols_to_check = ['system', 'sub_system', 'sub_sub_system']

# for col in cols_to_check:
#     count_col = f"{col}_char_count"
    
#     unique_df = tnm_hierarchy_df[[col]].drop_duplicates().dropna()
#     unique_df[count_col] = unique_df[col].astype(str).str.len()
    
#     print(f"--- Top 10 Longest {col} entries (TNM) ---")
#     display(unique_df.nlargest(10, count_col)[[col, count_col]])
#     print("\n")

In [70]:
tnm_cm_data = load_cm_data(cm_data_paths["TNM"])

tnm_cm_data.rename(columns={
    "Mapped_System": "system",
    "Mapped_Sub_System": "sub_system",
    "Mapped_Sub_Sub_System": "sub_sub_system",
    "LOCATION": "station",
    }, inplace=True)

for col in tnm_cm_data.columns:
    tnm_cm_data[col] = tnm_cm_data[col].replace("\n", " ", regex=True)
    tnm_cm_data[col] = tnm_cm_data[col].replace("\r", " ", regex=True)
    tnm_cm_data[col] = tnm_cm_data[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

tnm_cm_data['created_on'] = pd.to_datetime(
    tnm_cm_data['DATE'].astype(str) + ' ' + tnm_cm_data['TIME'].astype(str),
    format='mixed',
    errors='coerce'
)

col_order = ['work_order_no', 'notification_no', 'created_on', 'station', 'system', 'sub_system', 'sub_sub_system']
tnm_cm_data = tnm_cm_data[col_order]

tnm_cm_data['work_order_no'] = pd.to_numeric(tnm_cm_data['work_order_no'], errors='coerce').fillna(0).astype(int)
tnm_cm_data['notification_no'] = pd.to_numeric(tnm_cm_data['notification_no'], errors='coerce').fillna(0).astype(int)

tnm_cm_data = tnm_cm_data[tnm_cm_data['work_order_no'].notnull() & (tnm_cm_data['work_order_no'] != 0)]
tnm_cm_data_incomplete = tnm_cm_data[tnm_cm_data['work_order_no'].isnull() | (tnm_cm_data['work_order_no'] == 0)]

tnm_cm_data.head()

,work_order_no,notification_no,created_on,station,system,sub_system,sub_sub_system
0,4000449001,11892690,2022-01-02 00:04:46,DEPOT,SWITCH 3,Switchdeck & RC Beam,Flipper
1,4000444230,11895462,2022-01-07 10:38:43,DEPOT,SWITCH 5 & 6,Switchdeck & RC Beam,RC Beam
2,4000445244,11897330,2022-01-17 18:25:02,DEPOT,SWITCH 5 & 6,Switchdeck & RC Beam,RC Beam
3,4000470781,11910007,2022-02-16 16:51:21,DEPOT,Compressor 1 & 2,Filter,Filter To remove contaminants like liquid wate...
4,4000458883,11918502,2022-03-04 19:09:44,KLS,SWITCH 1 & 2,Switchdeck & RC Beam,Flipper


In [71]:
import numpy as np

checks = {
    'system': 'system',
    'sub_system': 'sub_system',
    'sub_sub_system': 'sub_sub_system',
}

for target_col, ref_col in checks.items():
    tnm_cm_data[target_col] = tnm_cm_data[target_col].replace("Others/Unmapped", "Others")
    valid_values = tnm_hierarchy_df[ref_col].unique()
    
    check_col_name = f'check_{target_col}'
    tnm_cm_data[check_col_name] = tnm_cm_data[target_col].isin(valid_values)
    
    num_false = (~tnm_cm_data[check_col_name]).sum()
    num_true = tnm_cm_data[check_col_name].sum()
    
    print(f"--- Results for {target_col} ---")
    print(f"Total rows: {len(tnm_cm_data)}")
    print(f"Valid: {num_true} | Invalid: {num_false}")
    if num_false > 0:
        print(f"Invalid samples: {tnm_cm_data.loc[~tnm_cm_data[check_col_name], target_col].unique()}")
    print("-" * 30)

tnm_hierarchy_df.to_csv("system_hierarchy/TNM_Hierarchy.csv", sep="|", index=False)
tnm_cm_data.to_csv("system_hierarchy/TNM_CM_Data.csv", sep="|", index=False)

--- Results for system ---
Total rows: 446
Valid: 446 | Invalid: 0
------------------------------
--- Results for sub_system ---
Total rows: 446
Valid: 446 | Invalid: 0
------------------------------
--- Results for sub_sub_system ---
Total rows: 446
Valid: 446 | Invalid: 0
------------------------------


### Combine all hierarchy

In [72]:
hierarchy_dfs = {
    "PSD": psd_hierarchy_df,
    "SNC": snc_hierarchy_df,
    "RSD": rsd_hierarchy_df,
    "TNM": tnm_hierarchy_df
}

# export all in one df
combined_hierarchy_df = pd.concat(hierarchy_dfs.values(), ignore_index=True)

# System DF
system_df = (combined_hierarchy_df[["system", "department"]]
             .replace(r'^\s*$', np.nan, regex=True)
             .dropna(subset=["system"])
             .drop_duplicates()
             .reset_index(drop=True))

# Sub-System DF
subsystem_df = (combined_hierarchy_df[["sub_system", "department"]]
                .replace(r'^\s*$', np.nan, regex=True)
                .dropna(subset=["sub_system"])
                .drop_duplicates()
                .reset_index(drop=True))

# Sub-Sub-System DF
subsubsystem_df = (combined_hierarchy_df[["sub_sub_system", "department"]]
                   .replace(r'^\s*$', np.nan, regex=True)
                   .dropna(subset=["sub_sub_system"])
                   .drop_duplicates()
                   .reset_index(drop=True))

# Root Cause DF
root_cause_df = (combined_hierarchy_df[["root_cause", "department"]]
                 .replace(r'^\s*$', np.nan, regex=True)
                 .dropna(subset=["root_cause"])
                 .drop_duplicates()
                 .reset_index(drop=True))

data_outputs = {
    "Combined_Hierarchy": combined_hierarchy_df,
    "System": system_df,
    "Sub_System": subsystem_df,
    "Sub_Sub_System": subsubsystem_df,
    "Root_Cause": root_cause_df
}

for name, df in data_outputs.items():
    file_path = f"system_hierarchy/separate/{name}.csv"
    df.to_csv(file_path, sep="|",index=False)
    print(f"Exported: {file_path}")

Exported: system_hierarchy/separate/Combined_Hierarchy.csv
Exported: system_hierarchy/separate/System.csv
Exported: system_hierarchy/separate/Sub_System.csv
Exported: system_hierarchy/separate/Sub_Sub_System.csv
Exported: system_hierarchy/separate/Root_Cause.csv
